<a id="de-casteljau"></a>

$$
\text{贝塞尔曲线 (Bezier Curve): 从线性插值出发} \\ \quad \\
\text{线性插值 (Linear Interpolation, lerp): } \text{lerp}(P_0, P_1, t) = (1 - t) P_0 + t P_1, \ t \in [0, 1] \text{, 就是一条线段} \\ \quad \\
\text{二次: 三个控制点, 做两层 lerp} \\
Q_0 = (1 - t) P_0 + t P_1, \quad Q_1 = (1 - t) P_1 + t P_2 \\
B(t) = (1 - t) Q_0 + t Q_1 = (1 - t)^2 P_0 + 2 t (1 - t) P_1 + t^2 P_2 \\ \quad \\
\text{德卡斯特里奥算法 (de Casteljau's Algorithm): n + 1 个控制点, 每一层对相邻两点做 lerp, 点数减 1, 做 n 层剩下 1 个点} \\
P_i^{(0)} = P_i, \quad P_i^{(r)} = (1 - t) P_i^{(r-1)} + t P_{i+1}^{(r-1)}, \quad B(t) = P_0^{(n)} \\ \quad \\
\text{性质:} \\
B(0) = P_0, \ B(1) = P_n \text{ (经过首尾两个控制点, 中间的控制点一般不在曲线上)} \\
\text{每一步都是凸组合 (权重非负, 和为 1), 所以曲线一定在控制点的凸包 (Convex Hull) 里}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb
from scipy import stats
from scipy.interpolate import BPoly, BSpline

np.set_printoptions(suppress=True, precision=4)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def de_casteljau(ctrl, t, return_levels=False):
    levels = [np.asarray(ctrl, dtype=float)]
    while len(levels[-1]) > 1:
        prev = levels[-1]
        levels.append((1 - t) * prev[:-1] + t * prev[1:])
    return (levels[-1][0], levels) if return_levels else levels[-1][0]

ctrl = np.array([[0.0, 0.0], [1.0, 3.0], [3.5, 3.2], [4.0, 0.5]])
ts = np.linspace(0, 1, 200)
curve = np.array([de_casteljau(ctrl, t) for t in ts])

# 二次的展开式
ctrl2 = ctrl[:3]
t0 = 0.37
quad_formula = (1 - t0)**2 * ctrl2[0] + 2 * t0 * (1 - t0) * ctrl2[1] + t0**2 * ctrl2[2]
print(f'二次: de Casteljau = {de_casteljau(ctrl2, t0)}, 展开式 = {quad_formula}')
print(f'B(0) = {curve[0]} == P0, B(1) = {curve[-1]} == P3')

t_show = 0.4
point, levels = de_casteljau(ctrl, t_show, return_levels=True)
plt.figure(figsize=(4, 3))
plt.plot(ctrl[:, 0], ctrl[:, 1], 'ko--', alpha=0.5, label='control polygon')
plt.plot(curve[:, 0], curve[:, 1], 'b', label='cubic Bezier')
for lv, color in zip(levels[1:-1], ['g', 'm']):
    plt.plot(lv[:, 0], lv[:, 1], 'o-', color=color, markersize=4)
plt.plot(*point, 'ro', label=f'B({t_show})')
plot_nicely()
plt.show()

<a id="bernstein"></a>

$$
\text{伯恩斯坦多项式 (Bernstein Polynomial) 形式} \\
B(t) = \sum_{i=0}^{n} b_{i,n}(t) \, P_i, \quad b_{i,n}(t) = \binom{n}{i} t^i (1 - t)^{n - i} \\ \quad \\
\text{推导: 在 de Casteljau 的三角形里, 每往下一层, 一个点的贡献要么 "留在原位" 被乘以 } (1 - t) \text{, 要么 "向左移一位" 被乘以 t} \\
P_i \text{ 要走到最底层的 } P_0^{(n)} \text{, n 步中必须正好有 i 步是 "移位" (乘 t), n - i 步是 "留下" (乘 1 - t)} \\
\text{每条路径的权重都是 } t^i (1 - t)^{n - i} \text{, 路径数 = 从 n 步里选 i 步 = } \binom{n}{i} \\
\text{这和二项分布的推导完全一样: } b_{i,n}(t) = P(X = i), \ X \sim \text{Binomial}(n, p = t) \\ \quad \\
\text{单位分解 (Partition of Unity), 二项式定理:} \\
\sum_{i=0}^{n} b_{i,n}(t) = \sum_{i=0}^{n} \binom{n}{i} t^i (1 - t)^{n - i} = (t + (1 - t))^n = 1 \\
\text{推论: 仿射不变性 (Affine Invariance), 对控制点做平移 / 旋转 / 缩放, 和对曲线做同样的变换结果相同} \\
\text{(因为 } \sum b_i (A P_i + c) = A \sum b_i P_i + c \sum b_i = A B(t) + c \text{, 用到了权重和为 1)}
$$

数学基础: [二项分布](../math_basics/statistics.ipynb#binomial)

In [ ]:
def bernstein(i, n, t):
    return comb(n, i) * t**i * (1 - t)**(n - i)

def bezier_bernstein(ctrl, t):
    n = len(ctrl) - 1
    t = np.atleast_1d(t)
    weights = np.column_stack([bernstein(i, n, t) for i in range(n + 1)])   # (len(t), n+1)
    return weights @ ctrl

curve_b = bezier_bernstein(ctrl, ts)
print(f'Bernstein 形式 == de Casteljau ? {np.allclose(curve_b, curve)}')
# scipy 的 BPoly 就是 Bernstein 基, 系数形状 (n+1, 区间数, 维数)
bp = BPoly(ctrl[:, None, :], [0, 1])
print(f'== scipy BPoly ? {np.allclose(bp(ts), curve)}')
n = 3
print(f'b_(i,n)(t) == 二项分布 PMF ? {np.allclose([bernstein(i, n, 0.3) for i in range(n + 1)], stats.binom.pmf(range(n + 1), n, 0.3))}')
print(f'权重和 = 1 ? {np.allclose(sum(bernstein(i, n, ts) for i in range(n + 1)), 1)}')

# 仿射不变性
A = np.array([[0.8, -0.6], [0.6, 0.8]]) @ np.diag([1.5, 0.7])
c = np.array([2.0, -1.0])
print(f'变换控制点后求曲线 == 对曲线变换 ? {np.allclose(bezier_bernstein(ctrl @ A.T + c, ts), curve @ A.T + c)}')

plt.figure(figsize=(4, 3))
for i in range(n + 1):
    plt.plot(ts, bernstein(i, n, ts), label=f'b_{i},3')
plt.plot(ts, sum(bernstein(i, n, ts) for i in range(n + 1)), 'k--', label='sum = 1')
plt.xlabel('t')
plot_nicely()
plt.show()

<a id="bezier-derivative"></a>

$$
\text{贝塞尔曲线的导数} \\
\text{先对基函数求导 (乘积法则):} \\
\frac{d}{dt} b_{i,n}(t) = \binom{n}{i} \left[ i \, t^{i-1} (1 - t)^{n - i} - (n - i) \, t^i (1 - t)^{n - i - 1} \right] \\
\text{利用组合数恒等式 } i \binom{n}{i} = n \binom{n-1}{i-1}, \ (n - i) \binom{n}{i} = n \binom{n-1}{i} \text{:} \\
\frac{d}{dt} b_{i,n}(t) = n \left[ b_{i-1, n-1}(t) - b_{i, n-1}(t) \right] \quad (\text{约定 } b_{-1,n-1} = b_{n,n-1} = 0) \\
B'(t) = \sum_{i=0}^{n} n [b_{i-1,n-1} - b_{i,n-1}] P_i \text{, 按 } b_{j,n-1} \text{ 重新整理 (第一项令 j = i - 1):} \\
B'(t) = n \sum_{j=0}^{n-1} b_{j,n-1}(t) \, (P_{j+1} - P_j) \\
\text{导数也是一条贝塞尔曲线 (低一次), 控制点是相邻控制点的差乘以 n} \\ \quad \\
\text{端点切线: } B'(0) = n (P_1 - P_0), \ B'(1) = n (P_n - P_{n-1}) \\
\text{曲线在起点沿着 } P_0 \to P_1 \text{ 方向出发, 在终点沿着 } P_{n-1} \to P_n \text{ 方向到达}
$$

数学基础: [乘积法则](../math_basics/calculus.ipynb#derivative-rules) · [数值微分 (中心差分)](../math_basics/calculus.ipynb#numerical-diff)

In [ ]:
def bezier_derivative(ctrl, t):
    n = len(ctrl) - 1
    return n * bezier_bernstein(np.diff(ctrl, axis=0), t)

h = 1e-6
t_test = np.array([0.1, 0.5, 0.83])
numeric = (bezier_bernstein(ctrl, t_test + h) - bezier_bernstein(ctrl, t_test - h)) / (2 * h)
print(f'导数公式 == 中心差分 ? {np.allclose(bezier_derivative(ctrl, t_test), numeric, atol=1e-6)}')
print(f"B'(0) = {bezier_derivative(ctrl, 0.0)[0]}, 3 (P1 - P0) = {3 * (ctrl[1] - ctrl[0])}")
print(f"B'(1) = {bezier_derivative(ctrl, 1.0)[0]}, 3 (P3 - P2) = {3 * (ctrl[3] - ctrl[2])}")
print(f'== scipy BPoly.derivative ? {np.allclose(bp.derivative()(ts), bezier_derivative(ctrl, ts))}')

<a id="bezier-continuity"></a>

$$
\text{拼接多段贝塞尔曲线: 连续性 (Continuity)} \\
\text{第一段控制点 } P_0, P_1, P_2, P_3 \text{, 第二段 } Q_0, Q_1, Q_2, Q_3 \text{ (都是三次)} \\ \quad \\
C^0 \text{ (位置连续): } B_P(1) = B_Q(0) \iff P_3 = Q_0 \\
C^1 \text{ (切线连续): } B_P'(1) = B_Q'(0) \iff 3(P_3 - P_2) = 3(Q_1 - Q_0) \iff Q_1 = 2 P_3 - P_2 \\
\text{即 } P_2, P_3 = Q_0, Q_1 \text{ 三点共线, 并且 } P_3 \text{ 是中点 (矢量绘图软件里拖动锚点两侧的手柄就是这个约束)} \\
C^2 \text{ 还要求二阶导数相等, 对贝塞尔曲线约束太强, 这就是为什么要用 B 样条}
$$

In [ ]:
P_seg = ctrl.copy()
Q_bad = np.array([P_seg[3], [5.5, 2.5], [7.0, 3.0], [8.0, 0.0]])   # 只有 C0
Q_good = Q_bad.copy()
Q_good[1] = 2 * P_seg[3] - P_seg[2]                                  # C1

for name, Q in [('只满足 C0', Q_bad), ('满足 C1', Q_good)]:
    d_end = bezier_derivative(P_seg, 1.0)[0]
    d_start = bezier_derivative(Q, 0.0)[0]
    print(f'{name}: 第一段终点导数 = {d_end}, 第二段起点导数 = {d_start}, 相等 ? {np.allclose(d_end, d_start)}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, Q, title in [(axs[0], Q_bad, 'C0 only (kink)'), (axs[1], Q_good, 'C1 (smooth)')]:
    for seg, color in [(P_seg, 'b'), (Q, 'r')]:
        cv = bezier_bernstein(seg, ts)
        ax.plot(cv[:, 0], cv[:, 1], color)
        ax.plot(seg[:, 0], seg[:, 1], 'o--', color=color, alpha=0.4, markersize=4)
    ax.set_title(title)
    ax.set_aspect('equal')
plot_nicely()
plt.show()

<a id="bspline"></a>

$$
\text{均匀三次 B 样条 (Uniform Cubic B-Spline)} \\
\text{目标: 所有分段都用同一组基函数 (平移), 段与段之间自动 } C^2 \text{ 连续, 每段只受 4 个相邻控制点影响 (局部控制)} \\ \quad \\
\text{推导: 设一个基函数 N(x) 在 [0, 4] 上非零, 由 4 段三次多项式组成, 第 k 段 (} x \in [k, k+1] \text{, 局部参数 } t = x - k \text{)} \\
N_k(t) = c_{k0} + c_{k1} t + c_{k2} t^2 + c_{k3} t^3, \quad k = 0, 1, 2, 3 \quad (16 \text{ 个未知数}) \\
\text{条件 (要求 N 在整条实轴上 } C^2 \text{, 在 [0, 4] 之外为 0):} \\
x = 0 \text{: } N_0(0) = N_0'(0) = N_0''(0) = 0 \quad (3 \text{ 个}) \\
x = 1, 2, 3 \text{: } N_{k-1}^{(m)}(1) = N_k^{(m)}(0), \ m = 0, 1, 2 \quad (9 \text{ 个}) \\
x = 4 \text{: } N_3(1) = N_3'(1) = N_3''(1) = 0 \quad (3 \text{ 个}) \\
\text{归一化 (单位分解): 在 t = 0 处 } N_0(0) + N_1(0) + N_2(0) + N_3(0) = 1 \quad (1 \text{ 个}) \\
16 \text{ 个线性方程解 16 个未知数, 结果:} \\
N_0 = \frac{t^3}{6}, \quad N_1 = \frac{-3t^3 + 3t^2 + 3t + 1}{6}, \quad N_2 = \frac{3t^3 - 6t^2 + 4}{6}, \quad N_3 = \frac{(1 - t)^3}{6} \\ \quad \\
\text{一段曲线 (控制点 } P_i, \cdots, P_{i+3} \text{), 写成矩阵形式:} \\
S_i(t) = \begin{bmatrix} t^3 & t^2 & t & 1 \end{bmatrix} \frac{1}{6} \begin{bmatrix} -1 & 3 & -3 & 1 \\ 3 & -6 & 3 & 0 \\ -3 & 0 & 3 & 0 \\ 1 & 4 & 1 & 0 \end{bmatrix} \begin{bmatrix} P_i \\ P_{i+1} \\ P_{i+2} \\ P_{i+3} \end{bmatrix} \\
\text{(} P_i \text{ 的权重是 } N_3 \text{, } P_{i+3} \text{ 的权重是 } N_0 \text{: 离当前段越远的控制点, 用的是基函数越靠边的那一段)} \\ \quad \\
\text{检查连续性: 第 i 段的终点 } S_i(1) = \frac{P_{i+1} + 4 P_{i+2} + P_{i+3}}{6} = S_{i+1}(0) \\
\text{B 样条一般不经过控制点, 换来的是 } C^2 \text{ 连续和局部控制}
$$

数学基础: [解线性方程组](../math_basics/linear_algebra.ipynb#inverse) · [矩阵的秩](../math_basics/linear_algebra.ipynb#rank)

In [ ]:
# 解 16 个线性方程, 未知数顺序 c[k, m] (第 k 段, t^m 的系数) -> 下标 4k + m
def poly_row(k, t, deriv):
    # 返回第 k 段在 t 处的 deriv 阶导数对 16 个系数的线性组合
    row = np.zeros(16)
    for m in range(4):
        if m >= deriv:
            coef = np.prod(range(m - deriv + 1, m + 1)) if deriv > 0 else 1   # m (m-1) ... (m-deriv+1)
            row[4 * k + m] = coef * t**(m - deriv)
    return row

rows, rhs = [], []
for d in range(3):
    rows.append(poly_row(0, 0.0, d)); rhs.append(0)            # x = 0
    rows.append(poly_row(3, 1.0, d)); rhs.append(0)            # x = 4
    for k in range(1, 4):
        rows.append(poly_row(k - 1, 1.0, d) - poly_row(k, 0.0, d)); rhs.append(0)   # x = 1, 2, 3
rows.append(sum(poly_row(k, 0.0, 0) for k in range(4))); rhs.append(1)
A_sys = np.array(rows)
print(f'方程组: {A_sys.shape}, 秩 = {np.linalg.matrix_rank(A_sys)}')
coef = np.linalg.solve(A_sys, rhs).reshape(4, 4)               # coef[k] = [c0, c1, c2, c3]
print(f'解出的 6 * 系数 (每行 [1, t, t^2, t^3]): \n{6 * coef}')

expected = np.array([[0, 0, 0, 1], [1, 3, 3, -3], [4, 0, -6, 3], [1, -3, 3, -1]]) / 6
print(f'== 推导的 N0, N1, N2, N3 ? {np.allclose(coef, expected)}')

M_bs = np.array([[-1, 3, -3, 1], [3, -6, 3, 0], [-3, 0, 3, 0], [1, 4, 1, 0]]) / 6
# 矩阵 M 的第 j 列 (t^3, t^2, t, 1 的系数) 对应控制点 P_(i+j), 它的权重是 N_(3-j)
M_from_coef = np.column_stack([coef[3 - j][::-1] for j in range(4)])
print(f'基矩阵 M == 由解出的 N 拼成的矩阵 ? {np.allclose(M_bs, M_from_coef)}')
tt = np.linspace(0, 1, 50)
print(f'单位分解对所有 t 成立 ? {np.allclose(sum(np.polyval(coef[k][::-1], tt) for k in range(4)), 1)}')

In [ ]:
def bspline_uniform(ctrl, samples_per_seg=50):
    T = np.linspace(0, 1, samples_per_seg)
    Tm = np.column_stack([T**3, T**2, T, np.ones_like(T)])
    segs = [Tm @ M_bs @ ctrl[i:i + 4] for i in range(len(ctrl) - 3)]
    return np.vstack(segs), segs

ctrl_bs = np.array([[0, 0], [1, 2.5], [2.5, 3], [4, 0.5], [5, 2], [6.5, 3], [7, 0.5], [8.5, 1.5]], dtype=float)
curve_bs, segs = bspline_uniform(ctrl_bs)

# scipy BSpline: 均匀节点 0, 1, ..., m + 3, 有效区间 [3, m]
m = len(ctrl_bs)
knots = np.arange(m + 4, dtype=float)
spl = BSpline(knots, ctrl_bs, 3)
x_eval = np.concatenate([i + 3 + np.linspace(0, 1, 50) for i in range(m - 3)])
print(f'矩阵形式 == scipy BSpline ? {np.allclose(curve_bs, spl(x_eval))}')

# 段与段之间的 0, 1, 2 阶导数连续
for d in range(3):
    jumps = [np.linalg.norm(spl.derivative(d)(j - 1e-9) - spl.derivative(d)(j + 1e-9)) if d else
             np.linalg.norm(segs[j - 4][-1] - segs[j - 3][0]) for j in range(4, m)]
    print(f'{d} 阶导数在接缝处的最大跳跃 = {max(jumps):.2e}')
print(f'第 0 段终点 = {segs[0][-1]}, (P1 + 4 P2 + P3) / 6 = {(ctrl_bs[1] + 4 * ctrl_bs[2] + ctrl_bs[3]) / 6}')

# 局部控制: 移动 P4 只影响用到它的 4 段
moved = ctrl_bs.copy()
moved[4] += [0, -2.0]
_, segs_moved = bspline_uniform(moved)
changed = [i for i in range(len(segs)) if not np.allclose(segs[i], segs_moved[i])]
print(f'移动 P4 后改变的段 = {changed} (用到 P4 的段: i = 1, 2, 3, 4)')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
x_basis = np.linspace(0, 4, 400)
N_x = np.select([x_basis < k + 1 for k in range(4)], [np.polyval(coef[k][::-1], x_basis - k) for k in range(4)])
axs[0].plot(x_basis, N_x, label='cubic B-spline basis N(x)')
for k in range(5):
    axs[0].axvline(k, color='gray', alpha=0.3)
axs[1].plot(ctrl_bs[:, 0], ctrl_bs[:, 1], 'ko--', alpha=0.4, label='control points')
axs[1].plot(curve_bs[:, 0], curve_bs[:, 1], 'b', label='B-spline')
axs[1].plot(*np.vstack(segs_moved).T, 'r:', label='after moving P4')
plot_nicely()
plt.show()